# Graph pooling — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## Pooling compresses many node representations into fewer graph-level representations
Graph pooling decides what information survives compression. The examples compare sum and mean readouts, assignment-based pooling, top-k pooling, and how a pooled adjacency records cross-cluster connectivity.

In [ ]:
# 1) sum pooling accumulates node evidence
X=np.array([[1.,0.],[0.,1.],[2.,1.],[1.,1.]]); s=X.sum(0)
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],s); plt.title('sum graph readout')
assert np.allclose(s,[4,3])

In [ ]:
# 2) mean pooling normalizes for graph size
X=np.array([[1.,0.],[0.,1.],[2.,1.],[1.,1.]]); m=X.mean(0)
plt.figure(figsize=(5,3)); plt.bar(['feat0','feat1'],m); plt.title('mean graph readout')
assert np.allclose(m,[1,0.75])

In [ ]:
# 3) assignment pooling maps 4 nodes to 2 clusters
X=np.array([[1.,0.],[0.,1.],[2.,1.],[1.,1.]]); S=np.array([[1,0],[1,0],[0,1],[0,1]],float); Xp=S.T@X
plt.figure(figsize=(5,3)); plt.imshow(Xp,cmap='Blues'); plt.colorbar(); plt.title('pooled node features S^T X')
assert np.allclose(Xp,[[1,1],[3,2]])

In [ ]:
# 4) top-k pooling keeps highest scoring nodes
X=np.array([[1.,0.],[0.,1.],[2.,1.],[1.,1.]]); scores=np.array([0.2,0.9,0.4,0.8]); keep=np.argsort(scores)[-2:]
plt.figure(figsize=(5,3)); plt.bar(range(4),scores,color=['tab:red' if i in keep else '0.7' for i in range(4)]); plt.title('top-k selected nodes')
assert set(keep.tolist())=={1,3}

In [ ]:
# 5) pooled adjacency counts edges between clusters
A=np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]],float); S=np.array([[1,0],[1,0],[0,1],[0,1]],float); Ap=S.T@A@S
plt.figure(figsize=(4,3)); plt.imshow(Ap,cmap='magma'); plt.colorbar(); plt.title('pooled adjacency')
assert np.allclose(Ap,[[2,2],[2,2]])